# IR Challenge v47 — Hard Domain Filter + 7-Signal Ensemble

**The breakthrough insight from teammate:**
97.6% of relevant papers are in the SAME scientific domain as the query.

**Algorithm:**
1. Restrict corpus to same-domain papers
2. Score with weighted blend of 7 signals (min-max normalized within domain)
3. Take top-90 from domain + 10 global fallback
4. CV-validated weights to avoid overfitting

**7 signals:**
- MiniLM cosine similarity (TA, cached)
- GTE-large cMaxSim — query TA vs body chunks (cached)
- BM25 on body chunks (built here)
- TF-IDF on TA (built here)
- BGE-large cosine similarity (TA, cached)
- Full-text TF-IDF 10K/2K (built here)
- ModernBERT full-text 8192-token (cached)

**Target:** training NDCG@10 > 0.73, held-out > 0.69

`Runtime -> T4 is fine`

---
## Step 1 - Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/ir_challenge')  # edit if needed
DATA_DIR  = PROJECT_DIR / 'data'
EMB_DIR   = DATA_DIR / 'embeddings'
SUBM_DIR  = PROJECT_DIR / 'submissions'
SUBM_DIR.mkdir(exist_ok=True)
for name in ['corpus.parquet','queries.parquet','qrels.json','held_out_queries.parquet']:
    p = DATA_DIR / name
    print(f"{'OK' if p.exists() else 'MISSING':8}  {p}")

OK        /content/drive/MyDrive/ir_challenge/data/corpus.parquet
OK        /content/drive/MyDrive/ir_challenge/data/queries.parquet
OK        /content/drive/MyDrive/ir_challenge/data/qrels.json
OK        /content/drive/MyDrive/ir_challenge/data/held_out_queries.parquet


---
## Step 2 - Install

In [3]:
!pip install -q rank_bm25 sentence-transformers
print('Done.')

Done.


---
## Step 3 - Imports & Helpers

In [4]:
import json, math, zipfile, re
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
TOP_K  = 100
DOM_K  = 90   # docs from same domain

def ndcg_at_k(ranked, relevant, k=10):
    dcg  = sum(1/math.log2(r+1) for r,d in enumerate(ranked[:k],1) if d in relevant)
    idcg = sum(1/math.log2(r+1) for r in range(1, min(len(relevant),k)+1))
    return dcg/idcg if idcg else 0.0

def recall_at_k(ranked, relevant, k=100):
    if not relevant: return 0.0
    return sum(1 for d in ranked[:k] if d in relevant) / len(relevant)

def average_precision(ranked, relevant):
    if not relevant: return 0.0
    hits = score = 0.0
    for rank, doc in enumerate(ranked, 1):
        if doc in relevant:
            hits += 1; score += hits/rank
    return score/len(relevant)

def quick_eval(submission, qrels, label='', verbose=True):
    ndcg_v, rec_v, map_v = [], [], []
    for qid, rel_list in qrels.items():
        rel    = set(rel_list)
        ranked = submission.get(qid, [])
        ndcg_v.append(ndcg_at_k(ranked, rel))
        rec_v.append(recall_at_k(ranked, rel))
        map_v.append(average_precision(ranked, rel))
    ndcg = float(np.mean(ndcg_v))
    if verbose:
        print(f'\n{chr(45)*48}\n  {label}\n{chr(45)*48}')
        print(f'  NDCG@10      {ndcg:.4f}')
        print(f'  Recall@100   {np.mean(rec_v):.4f}')
        print(f'  MAP          {np.mean(map_v):.4f}')
    return ndcg, float(np.mean(rec_v))

def load_npy_cache(cache_dir):
    cache_dir = Path(cache_dir)
    embs = np.load(cache_dir / 'embeddings.npy').astype('float32')
    with open(cache_dir / 'ids.json') as f: ids = json.load(f)
    print(f'  [cache] {cache_dir.name:<42} {embs.shape}')
    return embs, ids

def load_chunk_cache(cache_dir):
    cache_dir = Path(cache_dir)
    embs = np.load(cache_dir / 'chunk_embeddings.npy').astype('float32')
    idx  = np.load(cache_dir / 'paper_idx.npy')
    with open(cache_dir / 'ids.json') as f: ids = json.load(f)
    print(f'  [cache] {cache_dir.name:<42} chunks {embs.shape}')
    return embs, idx, ids

def minmax(arr):
    lo, hi = arr.min(), arr.max()
    return np.zeros_like(arr) if hi == lo else (arr - lo) / (hi - lo)

print('Helpers ready.')

Device: cuda
Helpers ready.


---
## Step 4 - Load Data

In [5]:
queries  = pd.read_parquet(DATA_DIR / 'queries.parquet')
corpus   = pd.read_parquet(DATA_DIR / 'corpus.parquet')
held_out = pd.read_parquet(DATA_DIR / 'held_out_queries.parquet')
with open(DATA_DIR / 'qrels.json') as f: qrels = json.load(f)

corpus_ids   = corpus['doc_id'].tolist()
train_ids    = queries['doc_id'].tolist()
held_out_ids = held_out['doc_id'].tolist()
corp_id2idx  = {d: i for i, d in enumerate(corpus_ids)}
n_docs       = len(corpus_ids)

train_doms   = queries['domain'].tolist()
held_doms    = held_out['domain'].tolist()

# Domain index: domain -> array of corpus indices
dom_to_cidx = {}
for i, row in corpus.iterrows():
    dom_to_cidx.setdefault(row['domain'], []).append(i)
dom_to_cidx = {k: np.array(v) for k, v in dom_to_cidx.items()}

# Verify domain overlap stat
same_dom = total = 0
for qid, qdom in zip(train_ids, train_doms):
    for rid in qrels.get(qid, []):
        cidx = corp_id2idx.get(rid)
        if cidx is not None:
            total += 1
            if corpus.iloc[cidx]['domain'] == qdom:
                same_dom += 1

print(f'Corpus {len(corpus):,}  Train {len(queries)}  Held-out {len(held_out)}')
print(f'Domains: {len(dom_to_cidx)}')
print(f'Relevant docs in same domain: {same_dom}/{total} = {same_dom/total:.1%}')
print(f'Cross-domain relevant docs  : {total - same_dom}')

Corpus 20,000  Train 100  Held-out 100
Domains: 19
Relevant docs in same domain: 718/736 = 97.6%
Cross-domain relevant docs  : 18


---
## Step 5 - Build BM25 on Body Chunks

BM25 on body section text (not title+abstract) is a strong signal.
Body text contains methods, datasets, related work — the citation signal.
Using chunk_meta to extract body sections.

In [6]:
def get_body_text(row):
    """Extract body text from full_text using chunk_meta."""
    full_text  = str(row.get('full_text', '') or '')
    chunk_meta = row.get('chunk_meta', '[]')
    try:
        meta = json.loads(chunk_meta) if isinstance(chunk_meta, str) else chunk_meta
    except Exception:
        return full_text[:5000]
    body_parts = []
    for i, entry in enumerate(meta):
        if entry.get('type') != 'body': continue
        start = entry['char_start']
        end   = meta[i+1]['char_start'] if i+1 < len(meta) else len(full_text)
        body_parts.append(full_text[start:end].strip())
    return ' '.join(body_parts) if body_parts else full_text[:5000]

def tokenize_bm25(text):
    return re.sub(r'[^a-zA-Z0-9]', ' ', str(text).lower()).split()

print('Extracting body text and building BM25 index ...')
corpus_body_texts = [get_body_text(row) for _, row in tqdm(corpus.iterrows(), total=n_docs)]
corpus_body_tokens = [tokenize_bm25(t) for t in tqdm(corpus_body_texts, desc='Tokenize')]

print('Building BM25 index ...')
bm25_body = BM25Okapi(corpus_body_tokens)
print('BM25 index built.')

# BM25 scores -> reciprocal rank matrices
print('Computing BM25 scores for training queries ...')
bm25_tr = np.zeros((n_docs, len(train_ids)), dtype=np.float32)
for qi, qid in enumerate(tqdm(train_ids, desc='BM25 train')):
    row    = queries[queries['doc_id'] == qid].iloc[0]
    tokens = tokenize_bm25(str(row.get('ta', '') or row.get('abstract', '')))
    scores = bm25_body.get_scores(tokens).astype('float32')
    # Convert to reciprocal rank
    order  = np.argsort(-scores)
    bm25_tr[order, qi] = 1.0 / (np.arange(n_docs) + 1)

print('Computing BM25 scores for held-out queries ...')
bm25_ho = np.zeros((n_docs, len(held_out_ids)), dtype=np.float32)
for qi, qid in enumerate(tqdm(held_out_ids, desc='BM25 held')):
    row    = held_out[held_out['doc_id'] == qid].iloc[0]
    tokens = tokenize_bm25(str(row.get('ta', '') or row.get('abstract', '')))
    scores = bm25_body.get_scores(tokens).astype('float32')
    order  = np.argsort(-scores)
    bm25_ho[order, qi] = 1.0 / (np.arange(n_docs) + 1)

# Quick eval
bm25_sub = {qid: [corpus_ids[i] for i in np.argsort(-bm25_tr[:, qi])[:TOP_K]]
            for qi, qid in enumerate(train_ids)}
quick_eval(bm25_sub, qrels, 'BM25 body chunks alone')
print('BM25 matrices ready.')

Extracting body text and building BM25 index ...


  0%|          | 0/20000 [00:00<?, ?it/s]

Tokenize:   0%|          | 0/20000 [00:00<?, ?it/s]

Building BM25 index ...
BM25 index built.
Computing BM25 scores for training queries ...


BM25 train:   0%|          | 0/100 [00:00<?, ?it/s]

Computing BM25 scores for held-out queries ...


BM25 held:   0%|          | 0/100 [00:00<?, ?it/s]


------------------------------------------------
  BM25 body chunks alone
------------------------------------------------
  NDCG@10      0.5018
  Recall@100   0.7193
  MAP          0.4003
BM25 matrices ready.


---
## Step 6 - Build TF-IDF Signals

In [7]:
ta_texts_corpus  = corpus['ta'].fillna('').tolist()
ta_texts_train   = queries['ta'].fillna('').tolist()
ta_texts_held    = held_out['ta'].fillna('').tolist()

ft_texts_corpus = corpus['full_text'].fillna('').str[:10000].tolist()
ft_texts_train  = queries['full_text'].fillna('').str[:2000].tolist()
ft_texts_held   = held_out['full_text'].fillna('').str[:2000].tolist()

# TA TF-IDF
print('Building TF-IDF TA ...')
tfidf_ta = TfidfVectorizer(
    sublinear_tf=True, ngram_range=(1,1),
    min_df=2, max_df=0.95, max_features=200_000, dtype=np.float32
)
corpus_ta_mat = tfidf_ta.fit_transform(ta_texts_corpus)
ta_tr = (corpus_ta_mat @ tfidf_ta.transform(ta_texts_train).T).toarray().astype(np.float32)
ta_ho = (corpus_ta_mat @ tfidf_ta.transform(ta_texts_held).T).toarray().astype(np.float32)

ta_sub = {qid: [corpus_ids[i] for i in np.argsort(-ta_tr[:, qi])[:TOP_K]]
          for qi, qid in enumerate(train_ids)}
quick_eval(ta_sub, qrels, 'TF-IDF TA alone')

# Full-text TF-IDF (10K corpus / 2K query)
print('\nBuilding Full-text TF-IDF ...')
tfidf_ft = TfidfVectorizer(
    sublinear_tf=True, ngram_range=(1,1),
    min_df=3, max_df=0.90, max_features=100_000, dtype=np.float32
)
corpus_ft_mat = tfidf_ft.fit_transform(ft_texts_corpus)
ft_tr = (corpus_ft_mat @ tfidf_ft.transform(ft_texts_train).T).toarray().astype(np.float32)
ft_ho = (corpus_ft_mat @ tfidf_ft.transform(ft_texts_held).T).toarray().astype(np.float32)

ft_sub = {qid: [corpus_ids[i] for i in np.argsort(-ft_tr[:, qi])[:TOP_K]]
          for qi, qid in enumerate(train_ids)}
quick_eval(ft_sub, qrels, 'Full-text TF-IDF (10K/2K) alone')

print('TF-IDF signals ready.')

Building TF-IDF TA ...

------------------------------------------------
  TF-IDF TA alone
------------------------------------------------
  NDCG@10      0.4904
  Recall@100   0.7407
  MAP          0.3779

Building Full-text TF-IDF ...

------------------------------------------------
  Full-text TF-IDF (10K/2K) alone
------------------------------------------------
  NDCG@10      0.5262
  Recall@100   0.8152
  MAP          0.4269
TF-IDF signals ready.


---
## Step 7 - Load Dense Embedding Signals

In [8]:
# MiniLM (384-dim)
MINI_DIR = EMB_DIR / 'sentence-transformers_all-MiniLM-L6-v2'
ml_c = np.load(MINI_DIR / 'corpus_embeddings.npy').astype('float32')
ml_q = np.load(MINI_DIR / 'query_embeddings.npy').astype('float32')
ml_h, _ = load_npy_cache(EMB_DIR / 'minilm_held_out')
ml_tr = (ml_c @ ml_q.T).astype('float32')   # (n_docs, n_train)
ml_ho = (ml_c @ ml_h.T).astype('float32')

ml_sub = {qid: [corpus_ids[i] for i in np.argsort(-ml_tr[:, qi])[:TOP_K]]
          for qi, qid in enumerate(train_ids)}
quick_eval(ml_sub, qrels, 'MiniLM alone')

# BGE-large (1024-dim)
bge_c, _ = load_npy_cache(EMB_DIR / 'bge_large_corpus')
bge_q, _ = load_npy_cache(EMB_DIR / 'bge_large_train_queries')
bge_h, _ = load_npy_cache(EMB_DIR / 'bge_large_held_queries')
bge_tr = (bge_c @ bge_q.T).astype('float32')
bge_ho = (bge_c @ bge_h.T).astype('float32')

bge_sub = {qid: [corpus_ids[i] for i in np.argsort(-bge_tr[:, qi])[:TOP_K]]
           for qi, qid in enumerate(train_ids)}
quick_eval(bge_sub, qrels, 'BGE-large alone')

# ModernBERT full-text (768-dim, 8192 tokens)
mb_c, _ = load_npy_cache(EMB_DIR / 'gte_modernbert_corpus_full')
mb_q, _ = load_npy_cache(EMB_DIR / 'gte_modernbert_train_full')
mb_h, _ = load_npy_cache(EMB_DIR / 'gte_modernbert_held_full')
mb_tr = (mb_c @ mb_q.T).astype('float32')
mb_ho = (mb_c @ mb_h.T).astype('float32')

mb_sub = {qid: [corpus_ids[i] for i in np.argsort(-mb_tr[:, qi])[:TOP_K]]
          for qi, qid in enumerate(train_ids)}
quick_eval(mb_sub, qrels, 'ModernBERT full-text alone')

# GTE-large cMaxSim (1024-dim corpus chunks)
gte_cce, gte_cci, _ = load_chunk_cache(EMB_DIR / 'gte_large_chunks_corpus')
gte_q_embs, _ = load_npy_cache(EMB_DIR / 'gte_large_train')
gte_h_embs, _ = load_npy_cache(EMB_DIR / 'gte_large_held')

print('Computing GTE cMaxSim ...')
ms_tr = np.zeros((n_docs, len(train_ids)), dtype='float32')
ms_ho = np.zeros((n_docs, len(held_out_ids)), dtype='float32')
sim_all_tr = gte_q_embs @ gte_cce.T
sim_all_ho = gte_h_embs @ gte_cce.T
for ci in tqdm(np.unique(gte_cci), desc='MaxSim train'):
    mask = gte_cci == ci
    ms_tr[ci, :] = sim_all_tr[:, mask].max(axis=1)
for ci in tqdm(np.unique(gte_cci), desc='MaxSim held'):
    mask = gte_cci == ci
    ms_ho[ci, :] = sim_all_ho[:, mask].max(axis=1)

ms_sub = {qid: [corpus_ids[i] for i in np.argsort(-ms_tr[:, qi])[:TOP_K]]
          for qi, qid in enumerate(train_ids)}
quick_eval(ms_sub, qrels, 'GTE-large cMaxSim alone')

print('\nAll signals ready.')

  [cache] minilm_held_out                            (100, 384)

------------------------------------------------
  MiniLM alone
------------------------------------------------
  NDCG@10      0.5073
  Recall@100   0.8104
  MAP          0.4080
  [cache] bge_large_corpus                           (20000, 1024)
  [cache] bge_large_train_queries                    (100, 1024)
  [cache] bge_large_held_queries                     (100, 1024)

------------------------------------------------
  BGE-large alone
------------------------------------------------
  NDCG@10      0.5580
  Recall@100   0.7989
  MAP          0.4498
  [cache] gte_modernbert_corpus_full                 (20000, 768)
  [cache] gte_modernbert_train_full                  (100, 768)
  [cache] gte_modernbert_held_full                   (100, 768)

------------------------------------------------
  ModernBERT full-text alone
------------------------------------------------
  NDCG@10      0.6111
  Recall@100   0.8645
  MAP     

MaxSim train:   0%|          | 0/20000 [00:00<?, ?it/s]

MaxSim held:   0%|          | 0/20000 [00:00<?, ?it/s]


------------------------------------------------
  GTE-large cMaxSim alone
------------------------------------------------
  NDCG@10      0.5361
  Recall@100   0.8254
  MAP          0.4290

All signals ready.


---
## Step 8 - Hard Domain Filter Retrieval

This is the key function. For each query:
1. Get all corpus papers in the SAME scientific domain
2. Score them with weighted blend of signals (min-max normalized per domain)
3. Take top DOM_K=90 from domain
4. Add TOP_K-DOM_K=10 global fallback docs for cross-domain coverage

This alone gives +0.11 NDCG@10.

In [9]:
# Collect all signal matrices
# train: (n_docs, n_train_queries)
# held:  (n_docs, n_held_queries)

SIGNAL_NAMES = ['MiniLM', 'MaxSim', 'BM25', 'TF-IDF TA', 'BGE-large', 'FT TF-IDF', 'ModernBERT']
ALL_TR = [ml_tr, ms_tr, bm25_tr, ta_tr, bge_tr, ft_tr, mb_tr]
ALL_HO = [ml_ho, ms_ho, bm25_ho, ta_ho, bge_ho, ft_ho, mb_ho]

print(f'Signal matrices: {len(ALL_TR)} signals')
for name, mat in zip(SIGNAL_NAMES, ALL_TR):
    print(f'  {name:<14}: {mat.shape}')


def hard_retrieve(qids, qdoms, sigs, weights, dom_k=DOM_K, total_k=TOP_K):
    """
    Hard domain filter retrieval.
    - Score within same-domain papers using weighted min-max normalized signals
    - Take top dom_k from domain
    - Fill remaining slots with global fallback
    """
    result = {}
    weights = np.array(weights, dtype='float32')
    weights = weights / weights.sum()  # normalize

    for qi, (qid, qdom) in enumerate(zip(qids, qdoms)):
        dom_idx = dom_to_cidx.get(qdom, np.arange(n_docs))

        # Score within domain
        scores = np.zeros(len(dom_idx), dtype='float32')
        for arr, w in zip(sigs, weights):
            col = arr[dom_idx, qi]
            scores += w * minmax(col)

        top_dom = dom_idx[np.argsort(-scores)[:min(dom_k, len(dom_idx))]]

        if len(top_dom) < total_k:
            # Global fallback for cross-domain coverage
            full_scores = np.zeros(n_docs, dtype='float32')
            for arr, w in zip(sigs, weights):
                full_scores += w * minmax(arr[:, qi])
            seen  = set(top_dom.tolist())
            extra = [j for j in np.argsort(-full_scores)
                     if j not in seen][:total_k - len(top_dom)]
            result[qid] = [corpus_ids[j] for j in list(top_dom) + extra][:total_k]
        else:
            result[qid] = [corpus_ids[j] for j in top_dom[:total_k]]
    return result


# Sanity check: single signal
sub_ml_dom = hard_retrieve(train_ids, train_doms, [ml_tr], [1.0])
quick_eval(sub_ml_dom, qrels, 'Hard domain + MiniLM only (sanity)')

Signal matrices: 7 signals
  MiniLM        : (20000, 100)
  MaxSim        : (20000, 100)
  BM25          : (20000, 100)
  TF-IDF TA     : (20000, 100)
  BGE-large     : (20000, 100)
  FT TF-IDF     : (20000, 100)
  ModernBERT    : (20000, 100)

------------------------------------------------
  Hard domain + MiniLM only (sanity)
------------------------------------------------
  NDCG@10      0.6423
  Recall@100   0.8863
  MAP          0.5493


(0.642327335679907, 0.8863240350446101)

---
## Step 9 - Dirichlet Weight Search

In [10]:
print(f'Searching optimal weights for {len(SIGNAL_NAMES)} signals ...')
print('Using 5000 Dirichlet random samples + refinement around best ...')

rng     = np.random.default_rng(42)
samples = rng.dirichlet(np.ones(len(ALL_TR)), size=5000)

best_ndcg = 0.0
best_w    = None
best_sub  = None

for i, ws in enumerate(tqdm(samples, desc='Dirichlet search')):
    sub = hard_retrieve(train_ids, train_doms, ALL_TR, ws)
    sc  = sum(ndcg_at_k(sub[qid], set(qrels.get(qid,[]))) for qid in train_ids) / len(train_ids)
    if sc > best_ndcg:
        best_ndcg = sc
        best_w    = ws.copy()

print(f'\nDirichlet best: NDCG@10={best_ndcg:.4f}')
print('Weights: ' + '  '.join(f'{n}={w:.3f}' for n, w in zip(SIGNAL_NAMES, best_w)))

# Fine-tune around best
print('\nFine-tuning ...')
delta = 0.04
n_sig = len(ALL_TR)
from itertools import product as iproduct

fine_vals = [[max(0, best_w[i]-delta), best_w[i], min(1, best_w[i]+delta)] for i in range(n_sig)]

for combo in tqdm(list(iproduct(*fine_vals)), desc='Fine-tune'):
    ws = np.array(combo, dtype='float32')
    if ws.sum() < 1e-6: continue
    ws = ws / ws.sum()
    sub = hard_retrieve(train_ids, train_doms, ALL_TR, ws)
    sc  = sum(ndcg_at_k(sub[qid], set(qrels.get(qid,[]))) for qid in train_ids) / len(train_ids)
    if sc > best_ndcg:
        best_ndcg = sc
        best_w    = ws.copy()

best_sub = hard_retrieve(train_ids, train_doms, ALL_TR, best_w)
ndcg_final, rec_final = quick_eval(best_sub, qrels, f'Best {len(SIGNAL_NAMES)}-sig (Dirichlet)')
print(f'\nFinal best: NDCG@10={ndcg_final:.4f}  Recall={rec_final:.4f}')
print('Weights: ' + '  '.join(f'{n}={w:.3f}' for n, w in zip(SIGNAL_NAMES, best_w)))

Searching optimal weights for 7 signals ...
Using 5000 Dirichlet random samples + refinement around best ...


Dirichlet search:   0%|          | 0/5000 [00:00<?, ?it/s]


Dirichlet best: NDCG@10=0.7458
Weights: MiniLM=0.056  MaxSim=0.021  BM25=0.087  TF-IDF TA=0.006  BGE-large=0.103  FT TF-IDF=0.165  ModernBERT=0.560

Fine-tuning ...


Fine-tune:   0%|          | 0/2187 [00:00<?, ?it/s]


------------------------------------------------
  Best 7-sig (Dirichlet)
------------------------------------------------
  NDCG@10      0.7495
  Recall@100   0.9242
  MAP          0.6604

Final best: NDCG@10=0.7495  Recall=0.9242
Weights: MiniLM=0.092  MaxSim=0.059  BM25=0.084  TF-IDF TA=0.045  BGE-large=0.061  FT TF-IDF=0.121  ModernBERT=0.539


---
## Step 10 - 5-Fold Cross-Validation

In [11]:
print('5-fold cross-validation to confirm no overfitting ...')
np.random.seed(42)
folds = np.array_split(np.random.permutation(100), 5)

cv_scores = []
for fold_i, val_idx in enumerate(folds):
    va_qids = [train_ids[i] for i in val_idx]
    va_doms = [train_doms[i] for i in val_idx]

    val_sub = hard_retrieve(va_qids, va_doms, ALL_TR, best_w)
    fold_q  = {qid: qrels[qid] for qid in va_qids if qid in qrels}
    fold_sc = sum(ndcg_at_k(val_sub[qid], set(fold_q.get(qid,[]))) for qid in va_qids) / len(va_qids)
    cv_scores.append(fold_sc)
    print(f'  Fold {fold_i+1}: NDCG@10={fold_sc:.4f}')

cv_mean = float(np.mean(cv_scores))
cv_std  = float(np.std(cv_scores))
print(f'\nCV mean  : {cv_mean:.4f} ± {cv_std:.4f}')
print(f'Training : {ndcg_final:.4f}')
print(f'Gap      : {ndcg_final - cv_mean:+.4f}')
if abs(ndcg_final - cv_mean) < 0.015:
    print('✓ CV ≈ Training — no overfitting, weights generalize.')
else:
    print('⚠ Gap > 0.015 — some overfitting, use CV weights as the safer estimate.')

5-fold cross-validation to confirm no overfitting ...
  Fold 1: NDCG@10=0.0000
  Fold 2: NDCG@10=0.0132
  Fold 3: NDCG@10=0.0000
  Fold 4: NDCG@10=0.0000
  Fold 5: NDCG@10=0.0517

CV mean  : 0.0130 ± 0.0200
Training : 0.7495
Gap      : +0.7365
⚠ Gap > 0.015 — some overfitting, use CV weights as the safer estimate.


---
## Step 11 - Per-Query Analysis

In [12]:
print('Per-query NDCG@10 breakdown ...')
per_q = []
for qi, (qid, qdom) in enumerate(zip(train_ids, train_doms)):
    rel    = set(qrels.get(qid, []))
    ranked = best_sub[qid]
    sc     = ndcg_at_k(ranked, rel)
    cross  = any(corpus.iloc[corp_id2idx[r]]['domain'] != qdom
                 for r in rel if r in corp_id2idx)
    best_r = next((r+1 for r, d in enumerate(ranked) if d in rel), 101)
    per_q.append({'ndcg': sc, 'best_rank': best_r, 'cross': cross,
                  'n_rel': len(rel), 'domain': qdom,
                  'title': str(queries.iloc[qi].get('title',''))[:50]})

per_q.sort(key=lambda x: x['ndcg'])
print(f'{'NDCG@10':>8} {'BestRank':>8} {'Cross':>5} {'nRel':>4}  Title')
print('  ' + chr(45)*65)
for r in per_q[:15]:
    print(f"{r['ndcg']:8.3f} {r['best_rank']:8d} {str(r['cross']):>5} {r['n_rel']:4d}  {r['title']}")

n_zero = sum(1 for r in per_q if r['ndcg'] == 0)
n_cross = sum(1 for r in per_q if r['cross'])
print(f'\nNDCG@10=0 queries : {n_zero}/100')
print(f'Cross-domain rels : {n_cross}/100 queries affected')

Per-query NDCG@10 breakdown ...
 NDCG@10 BestRank Cross nRel  Title
  -----------------------------------------------------------------
   0.000      101 False    2  Posttranscriptional Regulation of Insulin Resistan
   0.000       13 False    2  CardioID: Secure ECG-BCG Agnostic Interaction-Free
   0.141        9 False    3  Predicting Disease Related microRNA Based on Simil
   0.141        9 False    3  N-p-coumaroyloctopamine ameliorates hepatic glucos
   0.167        6  True    3  Does virtual currency development harm financial s
   0.193        8 False    2  Do Extraordinary Claims Require Extraordinary Evid
   0.195        3 False    4  Transcription Regulation of E-Cadherin by Zinc Fin
   0.264        4 False    2  SAC3: Reliable Hallucination Detection in Black-Bo
   0.330        5 False    3  Basal Bioenergetic Abnormalities in Skeletal Muscl
   0.384        2 False    5  Lipoteichoic Acid from Staphylococcus aureus Induc
   0.390        1 False    4  On the Inductive Bias of

---
## Step 12 - Build & Save Held-Out Submission

In [13]:
print('Building held-out submission ...')
ho_sub = hard_retrieve(held_out_ids, held_doms, ALL_HO, best_w)

assert len(set(held_out_ids) - set(ho_sub.keys())) == 0
assert all(len(v) == TOP_K for v in ho_sub.values())

json_path = SUBM_DIR / 'submission_data.json'
zip_path  = SUBM_DIR / 'submission.zip'
with open(json_path, 'w') as f: json.dump(ho_sub, f)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_path, arcname='submission_data.json')

print(f'Formula          : Hard domain filter + {len(SIGNAL_NAMES)}-signal ensemble')
print(f'Training NDCG@10 : {ndcg_final:.4f}')
print(f'Training Recall  : {rec_final:.4f}')
print(f'CV NDCG@10       : {cv_mean:.4f} ± {cv_std:.4f}')
print(f'Saved ZIP        : {zip_path}')
print('Done.')

Building held-out submission ...
Formula          : Hard domain filter + 7-signal ensemble
Training NDCG@10 : 0.7495
Training Recall  : 0.9242
CV NDCG@10       : 0.0130 ± 0.0200
Saved ZIP        : /content/drive/MyDrive/ir_challenge/submissions/submission.zip
Done.
